# Optimasi Pipeline ASR untuk Komputasi CPU Rendah-Sumber Daya
Notebook ini disusun ulang agar mengikuti pipeline penelitian 12 minggu untuk sistem ASR *offline-first* berbasis **Silero VAD + Distil-Whisper-Small ONNX INT8** pada CPU tanpa CUDA.

## Tujuan Notebook
1. Menyiapkan environment CPU-friendly
2. Menyiapkan dataset evaluasi dan preprocessing audio
3. Mengintegrasikan Silero VAD untuk trimming keheningan
4. Mengekspor model `distil-small.en` ke ONNX
5. Melakukan kuantisasi dinamis INT8
6. Menjalankan benchmark akurasi dan performa pada CPU
7. Menyediakan hook integrasi ke AIRA dan Gemini API


## Fase 1 - Setup Environment, Dataset, dan Preprocessing
Bagian ini menggantikan workflow fine-tuning LoRA/GPU pada notebook lama, karena target penelitian adalah **inferensi CPU rendah-sumber daya**, bukan training model.


In [ ]:
# Sel 1: Instalasi dependensi CPU-friendly untuk pipeline ASR penelitian
# Catatan: jalankan sekali saja. Hapus komentar pada baris %pip bila environment masih belum lengkap.
# %pip install -q onnxruntime optimum transformers datasets evaluate jiwer librosa soundfile openai-whisper huggingface_hub silero-vad psutil pandas matplotlib seaborn

print('Dependensi CPU-friendly siap digunakan.')


In [ ]:
# Sel 2: Import library dan konfigurasi global penelitian
import os
import gc
import time
import json
import math
import psutil
import shutil
import platform
import warnings
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from tqdm.auto import tqdm
from datasets import load_dataset, Audio
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, pipeline
from optimum.onnxruntime import ORTModelForSpeechSeq2Seq
import evaluate
import jiwer

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
ARTIFACT_DIR = PROJECT_ROOT / 'asr_cpu_artifacts'
AUDIO_DIR = ARTIFACT_DIR / 'audio_samples'
MODEL_DIR = ARTIFACT_DIR / 'models'
RESULT_DIR = ARTIFACT_DIR / 'results'

for target_dir in [ARTIFACT_DIR, AUDIO_DIR, MODEL_DIR, RESULT_DIR]:
    target_dir.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'distil-whisper/distil-small.en'
SAMPLE_RATE = 16000
NUM_EVAL_SAMPLES = 50
MAX_AUDIO_SECONDS = 30
LANGUAGE = 'en'
TASK = 'transcribe'
DEVICE = 'cpu'
CPU_PROVIDER = 'CPUExecutionProvider'

system_info = {
    'processor': platform.processor(),
    'machine': platform.machine(),
    'platform': platform.platform(),
    'python': platform.python_version(),
    'physical_cores': psutil.cpu_count(logical=False),
    'logical_cores': psutil.cpu_count(logical=True),
    'ram_gb': round(psutil.virtual_memory().total / (1024 ** 3), 2)
}

print(pd.DataFrame([system_info]))


In [ ]:
# Sel 3: Definisikan target eksperimen dan metrik penelitian
TARGETS_DF = pd.DataFrame([
    {'metric': 'WER drift vs baseline FP32', 'target': '<= 4 percentage points'},
    {'metric': 'Real-Time Factor', 'target': '<= 0.3'},
    {'metric': 'Model file size', 'target': '< 200 MB'},
    {'metric': 'Runtime RAM aktif', 'target': '< 1 GB'}
])
print(TARGETS_DF)


In [ ]:
# Sel 4: Muat dataset evaluasi dan batasi sampel agar benchmark tetap realistis di CPU
wer_metric = evaluate.load('wer')
raw_eval_ds = load_dataset('librispeech_asr', 'clean', split='validation')
raw_eval_ds = raw_eval_ds.select(range(NUM_EVAL_SAMPLES))
raw_eval_ds = raw_eval_ds.cast_column('audio', Audio(sampling_rate=SAMPLE_RATE))

print(raw_eval_ds)
print(raw_eval_ds[0])


In [ ]:
# Sel 5: Ubah dataset menjadi tabel metadata evaluasi
sample_rows = []
for row_idx in tqdm(range(len(raw_eval_ds))):
    item_obj = raw_eval_ds[row_idx]
    audio_info = item_obj['audio']
    audio_seconds = len(audio_info['array']) / audio_info['sampling_rate']
    sample_rows.append({
        'sample_id': row_idx,
        'speaker_id': item_obj.get('speaker_id', None),
        'chapter_id': item_obj.get('chapter_id', None),
        'text': item_obj['text'],
        'audio_seconds': audio_seconds
    })

eval_df = pd.DataFrame(sample_rows)
print(eval_df.head())

plt.figure(figsize=(8, 4))
sns.histplot(eval_df['audio_seconds'], bins=15, color='#1f77b4')
plt.title('Distribusi Durasi Audio Evaluasi')
plt.xlabel('Durasi audio dalam detik')
plt.ylabel('Jumlah sampel')
plt.show()


## Fase 2 - Integrasi Silero VAD, Ekspor ONNX, dan Kuantisasi INT8
Bagian ini menggantikan blok notebook lama yang berisi LoRA, PEFT, training, dan merge model. Fokus penelitian diarahkan ke **optimasi inferensi CPU**.


In [ ]:
# Sel 6: Processor baseline dan normalizer evaluasi
processor = AutoProcessor.from_pretrained(MODEL_ID)
normalizer = jiwer.Compose([
    jiwer.ToLowerCase(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
    jiwer.RemovePunctuation()
])

print('Processor berhasil dimuat untuk model ' + MODEL_ID)


In [ ]:
# Sel 7: Definisikan helper audio, metrik, dan monitor resource
process_obj = psutil.Process(os.getpid())

def normalize_text(text_value):
    if text_value is None:
        return ''
    return normalizer(text_value)

def get_memory_mb():
    return process_obj.memory_info().rss / (1024 ** 2)

def compute_rtf(inference_seconds, audio_seconds):
    if audio_seconds == 0:
        return np.nan
    return inference_seconds / audio_seconds

def save_wav_file(audio_array, sample_rate, file_path):
    sf.write(file_path, audio_array, sample_rate)
    return file_path


In [ ]:
# Sel 8: Integrasi Silero VAD untuk memotong keheningan sebelum decoding
try:
    from silero_vad import load_silero_vad, get_speech_timestamps
    vad_model = load_silero_vad()
    silero_ready = True
except Exception as exc_obj:
    vad_model = None
    silero_ready = False
    print('Silero VAD belum aktif: ' + str(exc_obj))

def apply_silero_vad(audio_array, sample_rate=SAMPLE_RATE, min_silence_ms=250):
    if not silero_ready:
        return audio_array, {'speech_ratio': 1.0, 'num_segments': 1, 'trimmed': False}
    audio_float32 = np.asarray(audio_array, dtype=np.float32)
    speech_segments = get_speech_timestamps(audio_float32, vad_model, sampling_rate=sample_rate, min_silence_duration_ms=min_silence_ms)
    if len(speech_segments) == 0:
        return audio_float32, {'speech_ratio': 1.0, 'num_segments': 0, 'trimmed': False}
    chunks = [audio_float32[seg['start']:seg['end']] for seg in speech_segments]
    merged_audio = np.concatenate(chunks) if len(chunks) > 0 else audio_float32
    speech_ratio = len(merged_audio) / max(len(audio_float32), 1)
    meta_obj = {'speech_ratio': speech_ratio, 'num_segments': len(speech_segments), 'trimmed': speech_ratio < 0.999}
    return merged_audio, meta_obj

print('Silero ready:', silero_ready)


In [ ]:
# Sel 9: Baseline pipeline FP32 CPU untuk pembanding akurasi dan kecepatan
baseline_model = AutoModelForSpeechSeq2Seq.from_pretrained(MODEL_ID)
baseline_pipe = pipeline(
    task='automatic-speech-recognition',
    model=baseline_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    device=-1
)
print('Baseline FP32 CPU pipeline siap.')


In [ ]:
# Sel 10: Benchmark baseline FP32 pada subset evaluasi
baseline_rows = []
for row_idx in tqdm(range(len(raw_eval_ds))):
    item_obj = raw_eval_ds[row_idx]
    audio_array = item_obj['audio']['array']
    audio_seconds = len(audio_array) / SAMPLE_RATE
    mem_before = get_memory_mb()
    start_time = time.perf_counter()
    pred_obj = baseline_pipe(audio_array.copy())
    infer_seconds = time.perf_counter() - start_time
    mem_after = get_memory_mb()
    baseline_rows.append({
        'sample_id': row_idx,
        'reference': normalize_text(item_obj['text']),
        'prediction': normalize_text(pred_obj['text']),
        'audio_seconds': audio_seconds,
        'inference_seconds': infer_seconds,
        'rtf': compute_rtf(infer_seconds, audio_seconds),
        'rss_delta_mb': mem_after - mem_before,
        'pipeline_name': 'baseline_fp32_cpu'
    })

baseline_result_df = pd.DataFrame(baseline_rows)
print(baseline_result_df.head())


In [ ]:
# Sel 11: Ringkas metrik baseline FP32
baseline_wer = wer_metric.compute(references=baseline_result_df['reference'].tolist(), predictions=baseline_result_df['prediction'].tolist())
baseline_summary_df = pd.DataFrame([{
    'pipeline_name': 'baseline_fp32_cpu',
    'wer': baseline_wer,
    'mean_rtf': baseline_result_df['rtf'].mean(),
    'median_rtf': baseline_result_df['rtf'].median(),
    'peak_rss_delta_mb': baseline_result_df['rss_delta_mb'].max(),
    'mean_audio_seconds': baseline_result_df['audio_seconds'].mean()
}])
print(baseline_summary_df)


In [ ]:
# Sel 12: Ekspor model Distil-Whisper ke ONNX menggunakan optimum-cli
ONNX_EXPORT_DIR = MODEL_DIR / 'distil_small_en_onnx'
ONNX_INT8_DIR = MODEL_DIR / 'distil_small_en_onnx_int8'

print('Folder ONNX export target: ' + str(ONNX_EXPORT_DIR))
print('Folder ONNX INT8 target: ' + str(ONNX_INT8_DIR))
print('Jalankan perintah shell berikut bila export belum dilakukan:')
print('optimum-cli export onnx --model ' + MODEL_ID + ' --task automatic-speech-recognition ' + str(ONNX_EXPORT_DIR))


In [ ]:
# Sel 13: Kuantisasi dinamis INT8 pada model ONNX
print('Jalankan setelah folder ONNX tersedia:')
print('optimum-cli onnxruntime quantize --onnx_model ' + str(ONNX_EXPORT_DIR) + ' --avx2 --per_channel ' + str(ONNX_INT8_DIR))


In [ ]:
# Sel 14: Muat model ONNX INT8 bila hasil export dan quantization sudah tersedia
onnx_ready = ONNX_INT8_DIR.exists()
if onnx_ready:
    ort_model = ORTModelForSpeechSeq2Seq.from_pretrained(str(ONNX_INT8_DIR), provider=CPU_PROVIDER)
    onnx_processor = AutoProcessor.from_pretrained(str(ONNX_INT8_DIR))
    onnx_pipe = pipeline(
        task='automatic-speech-recognition',
        model=ort_model,
        tokenizer=onnx_processor.tokenizer,
        feature_extractor=onnx_processor.feature_extractor,
        device=-1
    )
    print('Pipeline ONNX INT8 siap.')
else:
    ort_model = None
    onnx_processor = None
    onnx_pipe = None
    print('Folder ONNX INT8 belum tersedia. Selesaikan Sel 12-13 terlebih dahulu.')


## Fase 3 - Benchmarking WER, RTF, CPU RAM, Integrasi AIRA dan Gemini API


In [ ]:
# Sel 15: Benchmark pipeline ONNX INT8 dengan dan tanpa Silero VAD
benchmark_rows = []
if onnx_pipe is not None:
    for row_idx in tqdm(range(len(raw_eval_ds))):
        item_obj = raw_eval_ds[row_idx]
        source_audio = item_obj['audio']['array']
        source_seconds = len(source_audio) / SAMPLE_RATE
        for use_vad in [False, True]:
            audio_input = source_audio.copy()
            vad_meta = {'speech_ratio': 1.0, 'num_segments': 1, 'trimmed': False}
            if use_vad:
                audio_input, vad_meta = apply_silero_vad(audio_input, SAMPLE_RATE)
            mem_before = get_memory_mb()
            start_time = time.perf_counter()
            pred_obj = onnx_pipe(audio_input)
            infer_seconds = time.perf_counter() - start_time
            mem_after = get_memory_mb()
            benchmark_rows.append({
                'sample_id': row_idx,
                'pipeline_name': 'onnx_int8_vad' if use_vad else 'onnx_int8_no_vad',
                'reference': normalize_text(item_obj['text']),
                'prediction': normalize_text(pred_obj['text']),
                'audio_seconds_original': source_seconds,
                'audio_seconds_processed': len(audio_input) / SAMPLE_RATE,
                'speech_ratio': vad_meta['speech_ratio'],
                'num_segments': vad_meta['num_segments'],
                'inference_seconds': infer_seconds,
                'rtf': compute_rtf(infer_seconds, source_seconds),
                'rss_delta_mb': mem_after - mem_before
            })
benchmark_result_df = pd.DataFrame(benchmark_rows)
print(benchmark_result_df.head())


In [ ]:
# Sel 16: Ringkas seluruh hasil benchmarking
summary_rows = []
summary_rows.extend(baseline_summary_df.to_dict(orient='records'))
if len(benchmark_rows) > 0:
    for pipeline_name, part_df in benchmark_result_df.groupby('pipeline_name'):
        pipeline_wer = wer_metric.compute(references=part_df['reference'].tolist(), predictions=part_df['prediction'].tolist())
        summary_rows.append({
            'pipeline_name': pipeline_name,
            'wer': pipeline_wer,
            'mean_rtf': part_df['rtf'].mean(),
            'median_rtf': part_df['rtf'].median(),
            'peak_rss_delta_mb': part_df['rss_delta_mb'].max(),
            'mean_audio_seconds': part_df['audio_seconds_original'].mean()
        })
summary_df = pd.DataFrame(summary_rows)
print(summary_df)


In [ ]:
# Sel 17: Visualisasi perbandingan WER dan RTF
if 'summary_df' in globals() and len(summary_df) > 0:
    fig_obj, axes_arr = plt.subplots(1, 2, figsize=(12, 4))
    sns.barplot(data=summary_df, x='pipeline_name', y='wer', ax=axes_arr[0], palette='Blues_r')
    axes_arr[0].set_title('Perbandingan WER')
    axes_arr[0].tick_params(axis='x', rotation=20)
    sns.barplot(data=summary_df, x='pipeline_name', y='mean_rtf', ax=axes_arr[1], palette='Greens_r')
    axes_arr[1].axhline(0.3, color='red', linestyle='--', label='Target RTF 0.3')
    axes_arr[1].set_title('Perbandingan Mean RTF')
    axes_arr[1].tick_params(axis='x', rotation=20)
    axes_arr[1].legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Sel 18: Evaluasi ukuran model dan status kepatuhan target penelitian
model_size_rows = []
for folder_name in ['distil_small_en_onnx', 'distil_small_en_onnx_int8']:
    folder_path = MODEL_DIR / folder_name
    if folder_path.exists():
        total_bytes = 0
        for path_obj in folder_path.rglob('*'):
            if path_obj.is_file():
                total_bytes += path_obj.stat().st_size
        model_size_rows.append({'artifact': folder_name, 'size_mb': total_bytes / (1024 ** 2)})
model_size_df = pd.DataFrame(model_size_rows)
print(model_size_df)


In [ ]:
# Sel 19: Simpan hasil benchmark untuk analisis skripsi dan paper
if 'summary_df' in globals():
    summary_path = RESULT_DIR / 'benchmark_summary.csv'
    summary_df.to_csv(summary_path, index=False)
    print(str(summary_path))
if 'baseline_result_df' in globals():
    baseline_path = RESULT_DIR / 'baseline_predictions.csv'
    baseline_result_df.to_csv(baseline_path, index=False)
    print(str(baseline_path))
if 'benchmark_result_df' in globals() and len(benchmark_result_df) > 0:
    benchmark_path = RESULT_DIR / 'onnx_benchmark_predictions.csv'
    benchmark_result_df.to_csv(benchmark_path, index=False)
    print(str(benchmark_path))


## Fase 4 - Analisis Pareto dan Integrasi Sistem


In [ ]:
# Sel 20: Plot Pareto sederhana antara WER dan RTF
if 'summary_df' in globals() and len(summary_df) > 0:
    plt.figure(figsize=(7, 5))
    sns.scatterplot(data=summary_df, x='mean_rtf', y='wer', hue='pipeline_name', s=140)
    plt.axvline(0.3, color='red', linestyle='--')
    plt.title('Kurva Pareto Sederhana WER vs Mean RTF')
    plt.xlabel('Mean RTF')
    plt.ylabel('WER')
    plt.show()


In [ ]:
# Sel 21: Template integrasi ke AIRA atau service eksternal seperti Gemini API
ASR_CONFIG = {
    'pipeline_name': 'onnx_int8_vad',
    'sample_rate': SAMPLE_RATE,
    'language': LANGUAGE,
    'task': TASK,
    'offline_first': True,
    'provider': CPU_PROVIDER
}
print(json.dumps(ASR_CONFIG, indent=2))

def transcribe_for_aira(audio_array, use_vad=True):
    if onnx_pipe is None:
        raise RuntimeError('Pipeline ONNX INT8 belum siap.')
    audio_input = np.asarray(audio_array, dtype=np.float32)
    vad_meta = {'speech_ratio': 1.0, 'num_segments': 1, 'trimmed': False}
    if use_vad:
        audio_input, vad_meta = apply_silero_vad(audio_input, SAMPLE_RATE)
    result_obj = onnx_pipe(audio_input)
    return {
        'text': result_obj['text'],
        'vad_meta': vad_meta,
        'config': ASR_CONFIG
    }
